In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
import matplotlib.pyplot as plt
import warnings
import pandas as pd
warnings.filterwarnings("ignore")

DB = "loan_risk"
BR = "bronze_loans"
SR = "silver_loans"

spark.sql(f"USE {DB}")
print("Config loaded")


In [0]:
df = spark.table(f"{DB}.{BR}")

COLS = ["loan_amnt", "funded_amnt", "term", "int_rate",
    "installment", "grade", "emp_length", "annual_inc",
    "loan_status", "dti", "revol_util", "total_acc",
    "addr_state", "purpose", "home_ownership"]

df = df.select(COLS)
print(f"Final Columns: {len(df.columns)}")
display(df.limit(5))

In [0]:
display(
    df.groupBy("loan_status").count().orderBy(F.desc("count"))
)

In [0]:
df = df.filter(F.col("loan_status").isin(["Fully Paid", "Charged Off", "default"]))

print("Converting to Pandas...")
pdf = df.toPandas()

# Remove % signs
pdf["int_rate"]   = pdf["int_rate"].astype(str).str.replace("%", "", regex=False).str.strip()
pdf["revol_util"] = pdf["revol_util"].astype(str).str.replace("%", "", regex=False).str.strip()

# Force numeric — dirty values become NaN
numeric_cols = ["loan_amnt", "funded_amnt", "installment",
                "annual_inc", "dti", "revol_util",
                "total_acc", "int_rate"]

for c in numeric_cols:
    pdf[c] = pd.to_numeric(pdf[c], errors="coerce")

# Fill nulls
for c in numeric_cols:
    pdf[c] = pdf[c].fillna(pdf[c].median())

pdf["emp_length"] = pdf["emp_length"].fillna("Unknown")

# Create target column
pdf["default"] = pdf["loan_status"].isin(["Charged Off", "Default"]).astype(int)

# Drop remaining nulls
pdf = pdf.dropna()

print(f"✅ Rows          : {len(pdf):,}")
print(f"✅ Default rate  : {pdf['default'].mean():.1%}")
print(f"✅ Nulls remaining: {pdf.isnull().sum().sum()}")

In [0]:
# Convert back to Spark
df = spark.createDataFrame(pdf)

# 1. Term as number
df = df.withColumn(
    "term_months",
    F.regexp_extract(F.col("term"), r"(\d+)", 1).cast(IntegerType())
)

# 2. Grade as number
df = df.withColumn(
    "grade_num",
    F.when(F.col("grade") == "A", 1)
     .when(F.col("grade") == "B", 2)
     .when(F.col("grade") == "C", 3)
     .when(F.col("grade") == "D", 4)
     .when(F.col("grade") == "E", 5)
     .when(F.col("grade") == "F", 6)
     .when(F.col("grade") == "G", 7)
     .otherwise(4)
)

# 3. Credit utilization
df = df.withColumn(
    "credit_utilization",
    F.least(F.col("revol_util") / 100.0, F.lit(1.0))
)

# 4. DTI ratio
df = df.withColumn(
    "dti_ratio",
    F.when(
        F.col("annual_inc") > 0,
        (F.col("installment") * 12) / F.col("annual_inc")
    ).otherwise(0.0)
)

# 5. Log transforms
df = df.withColumn("log_annual_inc", F.log1p(F.col("annual_inc")))
df = df.withColumn("log_loan_amnt",  F.log1p(F.col("loan_amnt")))

# 6. Risk score
df = df.withColumn(
    "risk_score",
    (F.when(F.col("int_rate") > 15, 2).otherwise(1)) +
    (F.when(F.col("dti") > 20, 2).otherwise(1)) +
    (F.when(F.col("credit_utilization") > 0.7, 2).otherwise(1)) +
    F.col("grade_num")
)

print(f"✅ Total columns: {len(df.columns)}")
display(df.limit(5))

In [0]:
# Save as Silver Delta table
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("loan_risk.silver_loans")
)

print(f"✅ Silver table saved!")

# Verify
silver = spark.table("loan_risk.silver_loans")
print(f"✅ Rows    : {silver.count():,}")
print(f"✅ Columns : {len(silver.columns)}")

# Show Delta history
display(spark.sql("DESCRIBE HISTORY loan_risk.silver_loans"))